<a href="https://colab.research.google.com/github/Pranayshukla0610/MLFlow/blob/main/AI_Dashboard_Generator_using_Gemini%2C_SQLite_%26_Plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy plotly google-generativeai openpyxl tabulate sqlalchemy ipywidgets

In [2]:
import pandas as pd
import numpy as np

import sqlite3
import json
import os
import shutil
from pathlib import Path

from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go

import google.generativeai as genai
from google.colab import files

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
print("Pandas Version :", pd.__version__)
print("NumPy Version :", np.__version__)
print("SQLite Ready")
print("Plotly Ready")
print("Gemini Library Ready")

Pandas Version : 2.2.2
NumPy Version : 2.0.2
SQLite Ready
Plotly Ready
Gemini Library Ready


In [4]:
folders = [
    "data",
    "database",
    "schema",
    "exports",
    "logs"
]

for folder in folders:
  os.makedirs(folder, exist_ok=True)

print('Project folders created successfully')

Project folders created successfully


In [5]:
print("="*60)

print("Project Structure")

print("="*60)

for folder in os.listdir():

    print(folder)

Project Structure
.config
exports
database
schema
logs
data
sample_data


In [6]:
print(os.getcwd())

/content


In [7]:
database_path = "database/retail_business.db"
connection = sqlite3.connect(database_path)
connection.close()
print("Database Created Successfully!")

Database Created Successfully!


In [8]:
os.listdir("database")

['retail_business.db']

In [9]:
connection = sqlite3.connect(database_path)
cursor = connection.cursor()
cursor.execute('Select sqlite_version();')
print(cursor.fetchone())
connection.close()

('3.37.2',)


In [10]:
import importlib
import google.generativeai as genai

importlib.reload(genai)

<module 'google.generativeai' (<google.colab._import_hooks._hook_injector.HookInjectorLoader object at 0x7d3414bbb440>)>

In [11]:
import google.generativeai as genai

print(genai.configure)

functools.partial(<function configure at 0x7d3414c045e0>, transport='rest', client_options={'api_endpoint': 'http://localhost:40147'})


In [12]:
import google.generativeai as genai

print(genai.__version__)

0.8.6


In [13]:
!pip show google-generativeai

Name: google-generativeai
Version: 0.8.6
Summary: Google Generative AI High level API client library and tools.
Home-page: https://github.com/google/generative-ai-python
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: google-ai-generativelanguage, google-api-core, google-api-python-client, google-auth, protobuf, pydantic, tqdm, typing-extensions
Required-by: 


In [14]:
import google.generativeai as genai
from getpass import getpass

# Ask for the API key securely
api_key = getpass("Enter your Gemini API Key: ")

# Configure Gemini
genai.configure(api_key=api_key)

print("Gemini configured successfully!")

Enter your Gemini API Key: ··········
Gemini configured successfully!


In [15]:
model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content("Hello")

print(response.text)

Hello! How can I help you today?


In [16]:
from google.colab import files
uploaded = files.upload()

Saving supply_chain_data.csv to supply_chain_data.csv
Saving marketing_campaign (1).csv to marketing_campaign (1).csv
Saving marketing_campaign.csv to marketing_campaign.csv
Saving Groceries_dataset.csv to Groceries_dataset.csv
Saving Sample - Superstore.csv to Sample - Superstore.csv


In [17]:
print("Uploaded Files")

print("="*50)

for file in uploaded.keys():

    print(file)

Uploaded Files
supply_chain_data.csv
marketing_campaign (1).csv
marketing_campaign.csv
Groceries_dataset.csv
Sample - Superstore.csv


In [18]:
import shutil
import os

os.makedirs("data", exist_ok=True)

for file in uploaded.keys():

    shutil.move(file, f"data/{file}")

print("Files moved successfully.")

Files moved successfully.


In [19]:
import os
print(os.listdir())

['.config', 'exports', 'database', 'schema', 'logs', 'data', 'sample_data']


In [20]:
print(os.listdir("data"))

['marketing_campaign (1).csv', 'Sample - Superstore.csv', 'supply_chain_data.csv', 'marketing_campaign.csv', 'Groceries_dataset.csv']


In [21]:
import pandas as pd

from pathlib import Path

In [22]:
data_path = Path("data")

datasets = {}

for csv_file in data_path.glob("*.csv"):

    try:

        df = pd.read_csv(

            csv_file,

            encoding="latin1"

        )

        datasets[csv_file.stem] = df

        print(f"{csv_file.name} imported successfully.")

    except Exception as e:

        print(f"Error importing {csv_file.name}")

        print(e)

marketing_campaign (1).csv imported successfully.
Sample - Superstore.csv imported successfully.
supply_chain_data.csv imported successfully.
marketing_campaign.csv imported successfully.
Groceries_dataset.csv imported successfully.


In [23]:
print(datasets.keys())

dict_keys(['marketing_campaign (1)', 'Sample - Superstore', 'supply_chain_data', 'marketing_campaign', 'Groceries_dataset'])


In [24]:
print("Total Datasets :", len(datasets))

Total Datasets : 5


In [25]:
for name, df in datasets.items():

    print("="*60)

    print(name)

    print(df.shape)

marketing_campaign (1)
(2240, 1)
Sample - Superstore
(9994, 21)
supply_chain_data
(100, 24)
marketing_campaign
(2240, 1)
Groceries_dataset
(38765, 3)


In [26]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    display(df.head())

marketing_campaign (1)


,ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response
0,5524;1957;Graduation;Single;58138;0;0;2012-09-...
1,2174;1954;Graduation;Single;46344;1;1;2014-03-...
2,4141;1965;Graduation;Together;71613;0;0;2013-0...
3,6182;1984;Graduation;Together;26646;1;0;2014-0...
4,5324;1981;PhD;Married;58293;1;0;2014-01-19;94;...


Sample - Superstore


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


supply_chain_data


,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


marketing_campaign


,ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response
0,5524;1957;Graduation;Single;58138;0;0;2012-09-...
1,2174;1954;Graduation;Single;46344;1;1;2014-03-...
2,4141;1965;Graduation;Together;71613;0;0;2013-0...
3,6182;1984;Graduation;Together;26646;1;0;2014-0...
4,5324;1981;PhD;Married;58293;1;0;2014-01-19;94;...


Groceries_dataset


,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [27]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    df.info()

    print()

marketing_campaign (1)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                                                                                                                                                                       Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                                                                                                                                                                                       --------------  ----- 
 0   ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Re

In [28]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    print(df.isnull().sum())

    print()

marketing_campaign (1)
ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response    0
dtype: int64

Sample - Superstore
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

supply_chain_data
Product type               0
SKU                        0
Price                      0
Availability               0
Number of products sold    

In [29]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    print("Duplicate Rows :", df.duplicated().sum())

marketing_campaign (1)
Duplicate Rows : 0
Sample - Superstore
Duplicate Rows : 0
supply_chain_data
Duplicate Rows : 0
marketing_campaign
Duplicate Rows : 0
Groceries_dataset
Duplicate Rows : 759


In [30]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    print(f"Rows : {df.shape[0]}")

    print(f"Columns : {df.shape[1]}")

marketing_campaign (1)
Rows : 2240
Columns : 1
Sample - Superstore
Rows : 9994
Columns : 21
supply_chain_data
Rows : 100
Columns : 24
marketing_campaign
Rows : 2240
Columns : 1
Groceries_dataset
Rows : 38765
Columns : 3


In [31]:
summary = pd.DataFrame({

    "Dataset":[name for name in datasets.keys()],

    "Rows":[df.shape[0] for df in datasets.values()],

    "Columns":[df.shape[1] for df in datasets.values()],

    "Missing Values":[df.isnull().sum().sum() for df in datasets.values()],

    "Duplicate Rows":[df.duplicated().sum() for df in datasets.values()]

})

summary

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,marketing_campaign (1),2240,1,0,0
1,Sample - Superstore,9994,21,0,0
2,supply_chain_data,100,24,0,0
3,marketing_campaign,2240,1,0,0
4,Groceries_dataset,38765,3,0,759


In [32]:
summary.to_csv(

    "exports/dataset_summary.csv",

    index=False

)

print("Summary saved successfully.")

Summary saved successfully.


In [33]:
import os

os.listdir("exports")

['dataset_summary.csv']

In [34]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [35]:
print("="*60)
print("Datasets Loaded")
print("="*60)

for dataset in datasets.keys():
    print(dataset)

Datasets Loaded
marketing_campaign (1)
Sample - Superstore
supply_chain_data
marketing_campaign
Groceries_dataset


In [36]:
for name, df in datasets.items():

    print("="*70)
    print(name)

    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

marketing_campaign (1)
Rows    : 2240
Columns : 1
Sample - Superstore
Rows    : 9994
Columns : 21
supply_chain_data
Rows    : 100
Columns : 24
marketing_campaign
Rows    : 2240
Columns : 1
Groceries_dataset
Rows    : 38765
Columns : 3


In [37]:
for name, df in datasets.items():

    print("="*70)
    print(name)

    print(df.columns.tolist())

marketing_campaign (1)
['ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response']
Sample - Superstore
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
supply_chain_data
['Product type', 'SKU', 'Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Customer demographics', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location', 'Lead time', 'Production volumes', 'Manufacturing lea

In [38]:
for name, df in datasets.items():

    print("="*70)
    print(name)

    display(df.dtypes)

marketing_campaign (1)


,0
ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response,object


Sample - Superstore


,0
Row ID,int64
Order ID,object
Order Date,object
Ship Date,object
Ship Mode,object
Customer ID,object
Customer Name,object
Segment,object
Country,object
City,object


supply_chain_data


,0
Product type,object
SKU,object
Price,float64
Availability,int64
Number of products sold,int64
Revenue generated,float64
Customer demographics,object
Stock levels,int64
Lead times,int64
Order quantities,int64


marketing_campaign


,0
ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response,object


Groceries_dataset


,0
Member_number,int64
Date,object
itemDescription,object


In [39]:
missing_summary = {}

for name, df in datasets.items():

    missing = df.isnull().sum()

    missing_summary[name] = missing

    print("="*70)

    print(name)

    display(missing[missing > 0])

marketing_campaign (1)


,0


Sample - Superstore


,0


supply_chain_data


,0


marketing_campaign


,0


Groceries_dataset


,0


In [40]:
for name, df in datasets.items():

    print("="*70)
    print(name)

    missing_percent = (df.isnull().sum()/len(df))*100

    display(missing_percent[missing_percent>0].sort_values(ascending=False))

marketing_campaign (1)


,0


Sample - Superstore


,0


supply_chain_data


,0


marketing_campaign


,0


Groceries_dataset


,0


In [41]:
for name, df in datasets.items():

    print("="*70)
    print(name)

    missing_percent = (df.isnull().sum()/len(df))*100

    display(missing_percent[missing_percent>0].sort_values(ascending=False))

marketing_campaign (1)


,0


Sample - Superstore


,0


supply_chain_data


,0


marketing_campaign


,0


Groceries_dataset


,0


In [42]:
duplicate_summary = []

for name, df in datasets.items():

    duplicate_summary.append({

        "Dataset": name,

        "Duplicate Rows": df.duplicated().sum()

    })

duplicates = pd.DataFrame(duplicate_summary)

duplicates

,Dataset,Duplicate Rows
0,marketing_campaign (1),0
1,Sample - Superstore,0
2,supply_chain_data,0
3,marketing_campaign,0
4,Groceries_dataset,759


In [43]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    df.info()

    print()

marketing_campaign (1)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                                                                                                                                                                       Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                                                                                                                                                                                       --------------  ----- 
 0   ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Re

In [44]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    numerical = df.select_dtypes(include=np.number).columns

    print(numerical.tolist())

marketing_campaign (1)
[]
Sample - Superstore
['Row ID', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']
supply_chain_data
['Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping costs', 'Lead time', 'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Defect rates', 'Costs']
marketing_campaign
[]
Groceries_dataset
['Member_number']


In [45]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    for column in df.columns:

        print(column, ":", df[column].nunique())

marketing_campaign (1)
ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response : 2240
Sample - Superstore
Row ID : 9994
Order ID : 5009
Order Date : 1237
Ship Date : 1334
Ship Mode : 4
Customer ID : 793
Customer Name : 793
Segment : 3
Country : 1
City : 531
State : 49
Postal Code : 631
Region : 4
Product ID : 1862
Category : 3
Sub-Category : 17
Product Name : 1850
Sales : 5825
Quantity : 14
Discount : 12
Profit : 7287
supply_chain_data
Product type : 3
SKU : 100
Price : 100
Availability : 63
Number of products sold : 96
Revenue generated : 100
Customer demographics : 4
Stock levels : 65
Lead times : 29
Order quantities : 61
Shipping times : 10
Shipping carriers : 3
Shipping costs : 1

In [46]:
for name, df in datasets.items():

    print("="*80)

    print(name)

    categorical = df.select_dtypes(include="object")

    for col in categorical.columns[:5]:

        print("\n",col)

        display(df[col].value_counts().head())

marketing_campaign (1)

 ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response


,count
ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response,
1448;1963;Master;Married;33562;1;2;2014-06-25;33;21;12;12;0;3;3;3;2;0;4;4;0;0;0;0;0;0;3;11;0,1
10659;1979;2n Cycle;Together;7500;1;0;2013-05-07;7;2;8;11;3;8;21;4;3;2;2;7;0;0;0;0;0;0;3;11;0,1
7366;1982;Master;Single;75777;0;0;2013-07-04;12;712;26;538;69;13;80;1;3;6;11;1;0;1;1;0;0;0;3;11;1,1
6261;1979;Graduation;Married;58025;0;1;2013-11-26;81;270;31;88;11;48;22;3;3;2;10;4;0;0;0;0;0;0;3;11;0,1
9246;1985;Master;Together;40101;1;0;2012-10-14;73;171;3;129;26;24;62;4;6;1;6;7;0;0;0;0;0;0;3;11;0,1


Sample - Superstore

 Order ID


,count
Order ID,
CA-2017-100111,14
CA-2017-157987,12
CA-2016-165330,11
US-2016-108504,11
US-2015-126977,10



 Order Date


...

,count
ï»¿ID;Year_Birth;Education;Marital_Status;Income;Kidhome;Teenhome;Dt_Customer;Recency;MntWines;MntFruits;MntMeatProducts;MntFishProducts;MntSweetProducts;MntGoldProds;NumDealsPurchases;NumWebPurchases;NumCatalogPurchases;NumStorePurchases;NumWebVisitsMonth;AcceptedCmp3;AcceptedCmp4;AcceptedCmp5;AcceptedCmp1;AcceptedCmp2;Complain;Z_CostContact;Z_Revenue;Response,
1448;1963;Master;Married;33562;1;2;2014-06-25;33;21;12;12;0;3;3;3;2;0;4;4;0;0;0;0;0;0;3;11;0,1
10659;1979;2n Cycle;Together;7500;1;0;2013-05-07;7;2;8;11;3;8;21;4;3;2;2;7;0;0;0;0;0;0;3;11;0,1
7366;1982;Master;Single;75777;0;0;2013-07-04;12;712;26;538;69;13;80;1;3;6;11;1;0;1;1;0;0;0;3;11;1,1
6261;1979;Graduation;Married;58025;0;1;2013-11-26;81;270;31;88;11;48;22;3;3;2;10;4;0;0;0;0;0;0;3;11;0,1
9246;1985;Master;Together;40101;1;0;2012-10-14;73;171;3;129;26;24;62;4;6;1;6;7;0;0;0;0;0;0;3;11;0,1


Groceries_dataset

 Date


,count
Date,
21-01-2015,96
21-07-2015,93
08-08-2015,92
29-11-2015,92
30-04-2015,91



 itemDescription


,count
itemDescription,
whole milk,2502
other vegetables,1898
rolls/buns,1716
soda,1514
yogurt,1334


In [47]:
for name, df in datasets.items():

    numeric = df.select_dtypes(include=np.number)

    if len(numeric.columns)>1:

        fig = px.imshow(

            numeric.corr(),

            text_auto=True,

            title=f"Correlation Matrix - {name}"

        )

        fig.show()

In [48]:
for name, df in datasets.items():

    numeric = df.select_dtypes(include=np.number)

    for column in numeric.columns:

        fig = px.histogram(

            df,

            x=column,

            title=f"{name} - {column}"

        )

        fig.show()

In [49]:
for name, df in datasets.items():

    numeric = df.select_dtypes(include=np.number)

    for column in numeric.columns:

        fig = px.box(

            df,

            y=column,

            title=f"{name} - {column}"

        )

        fig.show()

In [50]:
for name, df in datasets.items():

    fig = px.imshow(

        df.isnull(),

        title=f"Missing Values - {name}",

        aspect="auto"

    )

    fig.show()

In [51]:
report = []

for name, df in datasets.items():

    report.append({

        "Dataset":name,

        "Rows":df.shape[0],

        "Columns":df.shape[1],

        "Missing":df.isnull().sum().sum(),

        "Duplicates":df.duplicated().sum(),

        "Numeric Columns":len(df.select_dtypes(include=np.number).columns),

        "Categorical Columns":len(df.select_dtypes(include="object").columns)

    })

eda_report = pd.DataFrame(report)

eda_report

,Dataset,Rows,Columns,Missing,Duplicates,Numeric Columns,Categorical Columns
0,marketing_campaign (1),2240,1,0,0,0,1
1,Sample - Superstore,9994,21,0,0,6,15
2,supply_chain_data,100,24,0,0,15,9
3,marketing_campaign,2240,1,0,0,0,1
4,Groceries_dataset,38765,3,0,759,1,2


In [53]:
eda_report.to_csv(

    "exports/EDA_Report.csv",

    index=False

)

print("EDA Report Saved Successfully")

EDA Report Saved Successfully


In [54]:
from google.colab import files

files.download("exports/EDA_Report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
eda_report.to_csv(

    "exports/EDA_Report.csv",

    index=False

)

print("EDA Report Saved Successfully")

EDA Report Saved Successfully


In [56]:
import os

os.makedirs("clean_data", exist_ok=True)

print("clean_data folder created.")

clean_data folder created.


In [57]:
cleaned_datasets = {}

for name, df in datasets.items():

    cleaned_datasets[name] = df.copy()

print("Copied successfully.")

Copied successfully.


In [58]:
for name, df in cleaned_datasets.items():

    df.columns = (

        df.columns

        .str.strip()

        .str.lower()

        .str.replace(" ", "_")

        .str.replace("-", "_")

        .str.replace("/", "_")

        .str.replace("(", "")

        .str.replace(")", "")

    )

print("Column names standardized.")

Column names standardized.


In [59]:
for name, df in cleaned_datasets.items():

    before = len(df)

    df.drop_duplicates(inplace=True)

    after = len(df)

    print(name)

    print("Removed :", before-after)

marketing_campaign (1)
Removed : 0
Sample - Superstore
Removed : 0
supply_chain_data
Removed : 0
marketing_campaign
Removed : 0
Groceries_dataset
Removed : 759


In [60]:
for name, df in cleaned_datasets.items():

    numeric = df.select_dtypes(include=np.number).columns

    for column in numeric:

        df[column] = df[column].fillna(

            df[column].median()

        )

In [61]:
for name, df in cleaned_datasets.items():

    categorical = df.select_dtypes(include="object").columns

    for column in categorical:

        df[column] = df[column].fillna(

            df[column].mode()[0]

        )

In [62]:
for name, df in cleaned_datasets.items():

    print(name)

    print(df.isnull().sum().sum())

marketing_campaign (1)
0
Sample - Superstore
0
supply_chain_data
0
marketing_campaign
0
Groceries_dataset
0


In [63]:
for name, df in cleaned_datasets.items():

    for column in df.columns:

        if "date" in column:

            df[column] = pd.to_datetime(

                df[column],

                errors="coerce"

            )

/tmp/ipykernel_6508/1627243707.py:7: UserWarning:

Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [64]:
for name, df in cleaned_datasets.items():

    print("="*60)

    print(name)

    display(df.dtypes)

marketing_campaign (1)


,0
ï»¿id;year_birth;education;marital_status;income;kidhome;teenhome;dt_customer;recency;mntwines;mntfruits;mntmeatproducts;mntfishproducts;mntsweetproducts;mntgoldprods;numdealspurchases;numwebpurchases;numcatalogpurchases;numstorepurchases;numwebvisitsmonth;acceptedcmp3;acceptedcmp4;acceptedcmp5;acceptedcmp1;acceptedcmp2;complain;z_costcontact;z_revenue;response,object


Sample - Superstore


,0
row_id,int64
order_id,object
order_date,datetime64[ns]
ship_date,datetime64[ns]
ship_mode,object
customer_id,object
customer_name,object
segment,object
country,object
city,object


supply_chain_data


,0
product_type,object
sku,object
price,float64
availability,int64
number_of_products_sold,int64
revenue_generated,float64
customer_demographics,object
stock_levels,int64
lead_times,int64
order_quantities,int64


marketing_campaign


,0
ï»¿id;year_birth;education;marital_status;income;kidhome;teenhome;dt_customer;recency;mntwines;mntfruits;mntmeatproducts;mntfishproducts;mntsweetproducts;mntgoldprods;numdealspurchases;numwebpurchases;numcatalogpurchases;numstorepurchases;numwebvisitsmonth;acceptedcmp3;acceptedcmp4;acceptedcmp5;acceptedcmp1;acceptedcmp2;complain;z_costcontact;z_revenue;response,object


Groceries_dataset


,0
member_number,int64
date,datetime64[ns]
itemdescription,object


In [65]:
for name, df in cleaned_datasets.items():

    numeric = df.select_dtypes(include=np.number).columns

    for column in numeric:

        df = df[df[column] >= 0]

    cleaned_datasets[name] = df

In [66]:
outlier_report = []

for name, df in cleaned_datasets.items():

    numeric = df.select_dtypes(include=np.number)

    for column in numeric.columns:

        Q1 = numeric[column].quantile(0.25)

        Q3 = numeric[column].quantile(0.75)

        IQR = Q3-Q1

        lower = Q1-1.5*IQR

        upper = Q3+1.5*IQR

        outliers = df[
            (df[column]<lower) |
            (df[column]>upper)
        ]

        outlier_report.append({

            "Dataset":name,

            "Column":column,

            "Outliers":len(outliers)

        })

In [67]:
outliers = pd.DataFrame(outlier_report)

outliers.head()

,Dataset,Column,Outliers
0,Sample - Superstore,row_id,0
1,Sample - Superstore,postal_code,0
2,Sample - Superstore,sales,1005
3,Sample - Superstore,quantity,145
4,Sample - Superstore,discount,0


In [68]:
outliers.to_csv(

    "exports/outlier_report.csv",

    index=False

)

In [69]:
if "Sample - Superstore" in cleaned_datasets:

    df = cleaned_datasets["Sample - Superstore"]

    if {"sales","quantity"}.issubset(df.columns):

        df["average_sales_per_item"] = (

            df["sales"] /

            df["quantity"]

        )

    if {"sales","profit"}.issubset(df.columns):

        df["profit_margin"] = (

            df["profit"] /

            df["sales"]

        )*100

    cleaned_datasets["Sample - Superstore"] = df

In [70]:
for name, df in cleaned_datasets.items():

    categorical = df.select_dtypes(include="object").columns

    for column in categorical:

        df[column] = (

            df[column]

            .astype(str)

            .str.strip()

            .str.title()

        )

In [71]:
for name, df in cleaned_datasets.items():

    df.to_csv(

        f"clean_data/{name}.csv",

        index=False

    )

print("All cleaned datasets saved.")

All cleaned datasets saved.


In [72]:
import os

print(os.listdir("clean_data"))

['marketing_campaign (1).csv', 'Sample - Superstore.csv', 'supply_chain_data.csv', 'marketing_campaign.csv', 'Groceries_dataset.csv']


In [73]:
report = []

for name, df in cleaned_datasets.items():

    report.append({

        "Dataset":name,

        "Rows":df.shape[0],

        "Columns":df.shape[1],

        "Missing Values":df.isnull().sum().sum(),

        "Duplicate Rows":df.duplicated().sum()

    })

clean_report = pd.DataFrame(report)

clean_report

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,marketing_campaign (1),2240,1,0,0
1,Sample - Superstore,8123,23,0,0
2,supply_chain_data,100,24,0,0
3,marketing_campaign,2240,1,0,0
4,Groceries_dataset,38006,3,0,0


In [74]:
clean_report.to_csv(

    "exports/cleaning_report.csv",

    index=False

)

print("Cleaning report saved.")

Cleaning report saved.


In [75]:
from google.colab import files

files.download("exports/cleaning_report.csv")
files.download("exports/outlier_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [76]:
os.makedirs("database",exist_ok=True)
print("Database Folder Created:")

Database Folder Created:


In [77]:
database_path = "database/retail_database.db"

In [78]:
connection = sqlite3.connect(database_path)
print("Connected Successfully!")

Connected Successfully!


In [79]:
clean_path = Path("clean_data")
csv_files = list(clean_path.glob("*.csv"))
print(csv_files)

[PosixPath('clean_data/marketing_campaign (1).csv'), PosixPath('clean_data/Sample - Superstore.csv'), PosixPath('clean_data/supply_chain_data.csv'), PosixPath('clean_data/marketing_campaign.csv'), PosixPath('clean_data/Groceries_dataset.csv')]


In [80]:
for file in csv_files:

    table_name = (

        file.stem
        .replace(" ", "_")
        .replace("-", "_")
        .lower()
    )

    print(f"Importing {table_name}")

    df = pd.read_csv(file)

    df.to_sql(

        table_name,

        connection,

        if_exists="replace",

        index=False

    )

print("All datasets imported.")

Importing marketing_campaign_(1)
Importing sample___superstore
Importing supply_chain_data
Importing marketing_campaign
Importing groceries_dataset
All datasets imported.


In [81]:
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    """,
    connection
)

tables

,name
0,marketing_campaign_(1)
1,sample___superstore
2,supply_chain_data
3,marketing_campaign
4,groceries_dataset


In [82]:
print("Total Tables :", len(tables))

Total Tables : 5


In [84]:
df = pd.read_sql(

    f'SELECT * FROM "{table}" LIMIT 5',

    connection

)

In [87]:
summary = []

for table in tables["name"]:

    total = pd.read_sql(

        f"""

        SELECT COUNT(*) AS Total

        FROM "{table}"

        """,

        connection

    )

    summary.append({

        "Table":table,

        "Rows":total.iloc[0]["Total"]

    })

summary = pd.DataFrame(summary)

summary

,Table,Rows
0,marketing_campaign_(1),2240
1,sample___superstore,8123
2,supply_chain_data,100
3,marketing_campaign,2240
4,groceries_dataset,38006


In [89]:
for table in tables["name"]:

    print("="*60)

    print(table)

    columns = pd.read_sql(

        f"""

        PRAGMA table_info("{table}")

        """,

        connection

    )

    display(columns[["name","type"]])

marketing_campaign_(1)


,name,type
0,ï»¿id;year_birth;education;marital_status;inco...,TEXT


sample___superstore


,name,type
0,row_id,INTEGER
1,order_id,TEXT
2,order_date,TEXT
3,ship_date,TEXT
4,ship_mode,TEXT
5,customer_id,TEXT
6,customer_name,TEXT
7,segment,TEXT
8,country,TEXT
9,city,TEXT


supply_chain_data


,name,type
0,product_type,TEXT
1,sku,TEXT
2,price,REAL
3,availability,INTEGER
4,number_of_products_sold,INTEGER
5,revenue_generated,REAL
6,customer_demographics,TEXT
7,stock_levels,INTEGER
8,lead_times,INTEGER
9,order_quantities,INTEGER


marketing_campaign


,name,type
0,ï»¿id;year_birth;education;marital_status;inco...,TEXT


groceries_dataset


,name,type
0,member_number,INTEGER
1,date,TEXT
2,itemdescription,TEXT


In [91]:
query = """

SELECT *

FROM sample___superstore

LIMIT 10

"""

result = pd.read_sql(query, connection)

result

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,product_id,category,sub_category,product_name,sales,quantity,discount,profit,average_sales_per_item,profit_margin
0,1,Ca-2016-152156,2016-11-08,2016-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,...,Fur-Bo-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.960,2,0.0,41.9136,130.980,16.00
1,2,Ca-2016-152156,2016-11-08,2016-11-11,Second Class,Cg-12520,Claire Gute,Consumer,United States,Henderson,...,Fur-Ch-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.940,3,0.0,219.5820,243.980,30.00
2,3,Ca-2016-138688,2016-06-12,2016-06-16,Second Class,Dv-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Off-La-10000240,Office Supplies,Labels,Self-Adhesive Address Labels For Typewriters B...,14.620,2,0.0,6.8714,7.310,47.00
3,5,Us-2015-108966,2015-10-11,2015-10-18,Standard Class,So-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Off-St-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164,11.184,11.25
4,6,Ca-2014-115812,2014-06-09,2014-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Fur-Fu-10001487,Furniture,Furnishings,Eldon Expressions Wood And Plastic Desk Access...,48.860,7,0.0,14.1694,6.980,29.00
5,7,Ca-2014-115812,2014-06-09,2014-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Off-Ar-10002833,Office Supplies,Art,Newell 322,7.280,4,0.0,1.9656,1.820,27.00
6,8,Ca-2014-115812,2014-06-09,2014-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Tec-Ph-10002275,Technology,Phones,Mitel 5320 Ip Phone Voip Phone,907.152,6,0.2,90.7152,151.192,10.00
7,9,Ca-2014-115812,2014-06-09,2014-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Off-Bi-10003910,Office Supplies,Binders,Dxl Angle-View Binders With Locking Rings By S...,18.504,3,0.2,5.7825,6.168,31.25
8,10,Ca-2014-115812,2014-06-09,2014-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Off-Ap-10002892,Office Supplies,Appliances,Belkin F5C206Vtel 6 Outlet Surge,114.900,5,0.0,34.4700,22.980,30.00
9,11,Ca-2014-115812,2014-06-09,2014-06-14,Standard Class,Bh-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,Fur-Ta-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.184,9,0.2,85.3092,189.576,5.00


In [92]:
query = """

SELECT

category,

SUM(sales) AS TotalSales

FROM sample___superstore

GROUP BY category

ORDER BY TotalSales DESC

"""

pd.read_sql(query, connection)

,category,TotalSales
0,Technology,716941.1400
1,Office Supplies,627438.3570
2,Furniture,484114.2085


In [93]:
query = """

SELECT

customer_name,

SUM(sales) AS Sales

FROM sample___superstore

GROUP BY customer_name

ORDER BY Sales DESC

LIMIT 10

"""

pd.read_sql(query, connection)

,customer_name,Sales
0,Tamara Chand,19026.172
1,Raymond Buch,15106.062
2,Tom Ashbrook,14511.348
3,Adrian Barton,13690.150
4,Ken Lonsdale,13494.268
5,Sanjit Chand,13408.846
6,Hunter Lopez,12873.298
7,Christopher Conant,11711.180
8,Todd Sumrall,10864.938
9,Karen Ferguson,10604.266


In [94]:
query = """

SELECT

SUM(sales)

AS TotalSales

FROM sample___superstore

"""

pd.read_sql(query, connection)

,TotalSales
0,1.828494e+06


In [95]:
summary.to_csv(

    "exports/database_summary.csv",

    index=False

)

print("Database summary saved.")

Database summary saved.


In [96]:
from google.colab import files

files.download("exports/database_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [97]:
connection.close()

print("Database Closed Successfully")

Database Closed Successfully


In [98]:
import os

os.makedirs("schema", exist_ok=True)

print("Schema folder created.")

Schema folder created.


In [99]:
import sqlite3
import pandas as pd
import json

In [100]:
connection = sqlite3.connect("database/retail_database.db")

print("Connected Successfully")

Connected Successfully


In [101]:
tables = pd.read_sql("""

SELECT name

FROM sqlite_master

WHERE type='table'

""", connection)

tables

,name
0,marketing_campaign_(1)
1,sample___superstore
2,supply_chain_data
3,marketing_campaign
4,groceries_dataset


In [102]:
table_names = tables["name"].tolist()

table_names

['marketing_campaign_(1)',
 'sample___superstore',
 'supply_chain_data',
 'marketing_campaign',
 'groceries_dataset']

In [104]:
schema = {}

for table in table_names:

    columns = pd.read_sql(

        f'PRAGMA table_info("{table}")',

        connection

    )

    schema[table] = columns

In [105]:
for table in schema:

    print("="*70)

    print(table)

    display(schema[table])

marketing_campaign_(1)


,cid,name,type,notnull,dflt_value,pk
0,0,ï»¿id;year_birth;education;marital_status;inco...,TEXT,0,None,0


sample___superstore


,cid,name,type,notnull,dflt_value,pk
0,0,row_id,INTEGER,0,None,0
1,1,order_id,TEXT,0,None,0
2,2,order_date,TEXT,0,None,0
3,3,ship_date,TEXT,0,None,0
4,4,ship_mode,TEXT,0,None,0
5,5,customer_id,TEXT,0,None,0
6,6,customer_name,TEXT,0,None,0
7,7,segment,TEXT,0,None,0
8,8,country,TEXT,0,None,0
9,9,city,TEXT,0,None,0


supply_chain_data


,cid,name,type,notnull,dflt_value,pk
0,0,product_type,TEXT,0,None,0
1,1,sku,TEXT,0,None,0
2,2,price,REAL,0,None,0
3,3,availability,INTEGER,0,None,0
4,4,number_of_products_sold,INTEGER,0,None,0
5,5,revenue_generated,REAL,0,None,0
6,6,customer_demographics,TEXT,0,None,0
7,7,stock_levels,INTEGER,0,None,0
8,8,lead_times,INTEGER,0,None,0
9,9,order_quantities,INTEGER,0,None,0


marketing_campaign


,cid,name,type,notnull,dflt_value,pk
0,0,ï»¿id;year_birth;education;marital_status;inco...,TEXT,0,None,0


groceries_dataset


,cid,name,type,notnull,dflt_value,pk
0,0,member_number,INTEGER,0,None,0
1,1,date,TEXT,0,None,0
2,2,itemdescription,TEXT,0,None,0


In [106]:
schema_dict = {}

for table in table_names:

    columns = pd.read_sql(

        f'PRAGMA table_info("{table}")',

        connection

    )

    schema_dict[table] = []

    for _, row in columns.iterrows():

        schema_dict[table].append({

            "column": row["name"],

            "datatype": row["type"]

        })

In [107]:
schema_dict

{'marketing_campaign_(1)': [{'column': 'ï»¿id;year_birth;education;marital_status;income;kidhome;teenhome;dt_customer;recency;mntwines;mntfruits;mntmeatproducts;mntfishproducts;mntsweetproducts;mntgoldprods;numdealspurchases;numwebpurchases;numcatalogpurchases;numstorepurchases;numwebvisitsmonth;acceptedcmp3;acceptedcmp4;acceptedcmp5;acceptedcmp1;acceptedcmp2;complain;z_costcontact;z_revenue;response',
   'datatype': 'TEXT'}],
 'sample___superstore': [{'column': 'row_id', 'datatype': 'INTEGER'},
  {'column': 'order_id', 'datatype': 'TEXT'},
  {'column': 'order_date', 'datatype': 'TEXT'},
  {'column': 'ship_date', 'datatype': 'TEXT'},
  {'column': 'ship_mode', 'datatype': 'TEXT'},
  {'column': 'customer_id', 'datatype': 'TEXT'},
  {'column': 'customer_name', 'datatype': 'TEXT'},
  {'column': 'segment', 'datatype': 'TEXT'},
  {'column': 'country', 'datatype': 'TEXT'},
  {'column': 'city', 'datatype': 'TEXT'},
  {'column': 'state', 'datatype': 'TEXT'},
  {'column': 'postal_code', 'datatyp

In [108]:
with open(

    "schema/database_schema.json",

    "w"

) as file:

    json.dump(

        schema_dict,

        file,

        indent=4

    )

print("Schema Saved Successfully")

Schema Saved Successfully


In [109]:
import os

os.listdir("schema")

['database_schema.json']

In [110]:
with open(

    "schema/database_schema.json",

    "r"

) as file:

    database_schema = json.load(file)

database_schema

{'marketing_campaign_(1)': [{'column': 'ï»¿id;year_birth;education;marital_status;income;kidhome;teenhome;dt_customer;recency;mntwines;mntfruits;mntmeatproducts;mntfishproducts;mntsweetproducts;mntgoldprods;numdealspurchases;numwebpurchases;numcatalogpurchases;numstorepurchases;numwebvisitsmonth;acceptedcmp3;acceptedcmp4;acceptedcmp5;acceptedcmp1;acceptedcmp2;complain;z_costcontact;z_revenue;response',
   'datatype': 'TEXT'}],
 'sample___superstore': [{'column': 'row_id', 'datatype': 'INTEGER'},
  {'column': 'order_id', 'datatype': 'TEXT'},
  {'column': 'order_date', 'datatype': 'TEXT'},
  {'column': 'ship_date', 'datatype': 'TEXT'},
  {'column': 'ship_mode', 'datatype': 'TEXT'},
  {'column': 'customer_id', 'datatype': 'TEXT'},
  {'column': 'customer_name', 'datatype': 'TEXT'},
  {'column': 'segment', 'datatype': 'TEXT'},
  {'column': 'country', 'datatype': 'TEXT'},
  {'column': 'city', 'datatype': 'TEXT'},
  {'column': 'state', 'datatype': 'TEXT'},
  {'column': 'postal_code', 'datatyp

In [111]:
import pprint

pprint.pprint(database_schema)

{'groceries_dataset': [{'column': 'member_number', 'datatype': 'INTEGER'},
                       {'column': 'date', 'datatype': 'TEXT'},
                       {'column': 'itemdescription', 'datatype': 'TEXT'}],
 'marketing_campaign': [{'column': 'ï»¿id;year_birth;education;marital_status;income;kidhome;teenhome;dt_customer;recency;mntwines;mntfruits;mntmeatproducts;mntfishproducts;mntsweetproducts;mntgoldprods;numdealspurchases;numwebpurchases;numcatalogpurchases;numstorepurchases;numwebvisitsmonth;acceptedcmp3;acceptedcmp4;acceptedcmp5;acceptedcmp1;acceptedcmp2;complain;z_costcontact;z_revenue;response',
                         'datatype': 'TEXT'}],
 'marketing_campaign_(1)': [{'column': 'ï»¿id;year_birth;education;marital_status;income;kidhome;teenhome;dt_customer;recency;mntwines;mntfruits;mntmeatproducts;mntfishproducts;mntsweetproducts;mntgoldprods;numdealspurchases;numwebpurchases;numcatalogpurchases;numstorepurchases;numwebvisitsmonth;acceptedcmp3;acceptedcmp4;acceptedcmp5;ac

In [112]:
summary = []

for table in database_schema:

    summary.append({

        "Table": table,

        "Columns": len(database_schema[table])

    })

summary = pd.DataFrame(summary)

summary

,Table,Columns
0,marketing_campaign_(1),1
1,sample___superstore,23
2,supply_chain_data,24
3,marketing_campaign,1
4,groceries_dataset,3


In [113]:
summary.to_csv(

    "exports/schema_summary.csv",

    index=False

)

print("Schema Summary Saved")

Schema Summary Saved


In [114]:
from google.colab import files

files.download(

    "schema/database_schema.json"

)

files.download(

    "exports/schema_summary.csv"

)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [115]:
connection.close()

print("Database Closed")

Database Closed


In [116]:
!pip install -q google-generativeai

In [117]:
import google.generativeai as genai

import json

import sqlite3

import pandas as pd

from IPython.display import Markdown

In [118]:
from getpass import getpass

api_key = getpass("Enter Gemini API Key: ")

genai.configure(api_key=api_key)

print("Gemini Connected Successfully")

Enter Gemini API Key: ··········
Gemini Connected Successfully


In [119]:
with open(
    "schema/database_schema.json",
    "r"
) as file:

    schema = json.load(file)

print(schema.keys())

dict_keys(['marketing_campaign_(1)', 'sample___superstore', 'supply_chain_data', 'marketing_campaign', 'groceries_dataset'])


In [120]:
schema_prompt = ""

for table in schema:

    schema_prompt += f"\nTable: {table}\n"

    for column in schema[table]:

        schema_prompt += f"- {column['column']} ({column['datatype']})\n"

print(schema_prompt)


Table: marketing_campaign_(1)
- ï»¿id;year_birth;education;marital_status;income;kidhome;teenhome;dt_customer;recency;mntwines;mntfruits;mntmeatproducts;mntfishproducts;mntsweetproducts;mntgoldprods;numdealspurchases;numwebpurchases;numcatalogpurchases;numstorepurchases;numwebvisitsmonth;acceptedcmp3;acceptedcmp4;acceptedcmp5;acceptedcmp1;acceptedcmp2;complain;z_costcontact;z_revenue;response (TEXT)

Table: sample___superstore
- row_id (INTEGER)
- order_id (TEXT)
- order_date (TEXT)
- ship_date (TEXT)
- ship_mode (TEXT)
- customer_id (TEXT)
- customer_name (TEXT)
- segment (TEXT)
- country (TEXT)
- city (TEXT)
- state (TEXT)
- postal_code (INTEGER)
- region (TEXT)
- product_id (TEXT)
- category (TEXT)
- sub_category (TEXT)
- product_name (TEXT)
- sales (REAL)
- quantity (INTEGER)
- discount (REAL)
- profit (REAL)
- average_sales_per_item (REAL)
- profit_margin (REAL)

Table: supply_chain_data
- product_type (TEXT)
- sku (TEXT)
- price (REAL)
- availability (INTEGER)
- number_of_produc

In [121]:
system_prompt = f"""
You are an expert SQL developer.

Database Engine:
SQLite

Database Schema:

{schema_prompt}

Rules:

Generate only SQL.

Do not explain.

Do not use markdown.

Return only executable SQLite SQL.

Never invent table names.

Never invent column names.
"""

In [122]:
model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [124]:
question = "Show total sales by category"

In [125]:
response = model.generate_content(

    system_prompt +

    "\n\nUser Question:\n" +

    question

)

sql = response.text.strip()

print(sql)

SELECT category, SUM(sales) FROM sample___superstore GROUP BY category


In [126]:
connection = sqlite3.connect(

    "database/retail_database.db"

)

In [127]:
result = pd.read_sql(

    sql,

    connection

)

result

,category,SUM(sales)
0,Furniture,484114.2085
1,Office Supplies,627438.3570
2,Technology,716941.1400


In [128]:
def generate_sql(question):

    response = model.generate_content(

        system_prompt +

        "\n\nQuestion:\n" +

        question

    )

    sql = response.text

    sql = sql.replace("```sql","")

    sql = sql.replace("```","")

    sql = sql.strip()

    return sql

In [129]:
sql = generate_sql(

    "Top 10 customers by sales"

)

print(sql)

SELECT customer_name, SUM(sales) AS total_sales FROM sample___superstore GROUP BY customer_name ORDER BY total_sales DESC LIMIT 10


In [130]:
df = pd.read_sql(

    sql,

    connection

)

df.head()

,customer_name,total_sales
0,Tamara Chand,19026.172
1,Raymond Buch,15106.062
2,Tom Ashbrook,14511.348
3,Adrian Barton,13690.150
4,Ken Lonsdale,13494.268


In [131]:
questions = [

    "Top 10 customers by sales",

    "Total sales by state",

    "Average profit by category",

    "Monthly sales trend",

    "Top selling products"

]

for q in questions:

    print("="*70)

    print(q)

    sql = generate_sql(q)

    print(sql)

Top 10 customers by sales
SELECT customer_name, SUM(sales) AS total_sales
FROM sample___superstore
GROUP BY customer_name
ORDER BY total_sales DESC
LIMIT 10
Total sales by state
SELECT state, SUM(sales) FROM sample___superstore GROUP BY state
Average profit by category
SELECT category, AVG(profit) FROM sample___superstore GROUP BY category
Monthly sales trend
SELECT
  STRFTIME('%Y-%m', order_date) AS sales_month,
  SUM(sales) AS total_sales
FROM sample___superstore
GROUP BY
  sales_month
ORDER BY
  sales_month;
Top selling products
SELECT product_name, SUM(sales) AS total_sales
FROM sample___superstore
GROUP BY product_name
ORDER BY total_sales DESC


In [133]:
def ask_database(question):

    sql = generate_sql(question)

    print("=" * 80)
    print("Generated SQL")
    print("=" * 80)
    print(sql)

    try:

        df = pd.read_sql(sql, connection)

        return df

    except Exception as e:

        print(e)

        return None

In [134]:
answer = ask_database(

    "Show sales by region"

)

answer

Generated SQL
SELECT region, SUM(sales) FROM sample___superstore GROUP BY region


,region,SUM(sales)
0,Central,359957.2320
1,East,517917.2280
2,South,300086.7200
3,West,650532.5255


In [135]:
while True:

    question = input("Ask a business question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    result = ask_database(question)

    display(result)

Ask a business question (type 'exit' to quit): Total profit from sample__superstore
Generated SQL
SELECT SUM(profit) FROM sample___superstore


,SUM(profit)
0,442528.3074


Ask a business question (type 'exit' to quit): exit


In [136]:
api_key = getpass("Enter Gemini API Key: ")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini Connected Successfully")

Enter Gemini API Key: ··········
Gemini Connected Successfully


In [137]:
connection = sqlite3.connect(
    "database/retail_database.db"
)

print("SQLite Connected")

SQLite Connected


In [138]:
with open(
    "schema/database_schema.json",
    "r"
) as file:

    schema = json.load(file)

In [139]:
schema_text = ""

for table in schema:

    schema_text += f"\nTable: {table}\n"

    for column in schema[table]:

        schema_text += f"{column['column']} ({column['datatype']})\n"

In [140]:
SYSTEM_PROMPT = f"""
You are an expert SQLite SQL developer.

The database schema is:

{schema_text}

Rules:

1. Generate ONLY SQLite SQL.

2. Do not explain anything.

3. Do not use markdown.

4. Do not invent columns.

5. Do not invent tables.

6. Always return executable SQL.

7. Never return comments.

8. Use LIMIT whenever appropriate.

Return only SQL.
"""

In [141]:
def generate_sql(question):

    prompt = SYSTEM_PROMPT + f"""

User Question:

{question}

SQL:
"""

    response = model.generate_content(prompt)

    sql = response.text.strip()

    return sql

In [142]:
question = "Show total sales by category"

sql = generate_sql(question)

print(sql)

SELECT category, SUM(sales) FROM sample___superstore GROUP BY category


In [143]:
!pip install -q plotly

In [144]:
def show_kpis(df):

    numeric_columns = df.select_dtypes(include=np.number).columns

    if len(numeric_columns) == 0:

        print("No numeric columns found.")

        return

    print("="*60)

    print("KEY PERFORMANCE INDICATORS")

    print("="*60)

    for column in numeric_columns:

        print(f"\n{column}")

        print(f"Total   : {df[column].sum():,.2f}")

        print(f"Average : {df[column].mean():,.2f}")

        print(f"Maximum : {df[column].max():,.2f}")

        print(f"Minimum : {df[column].min():,.2f}")

In [145]:
result = ask_database(
    "Show total sales by category"
)

show_kpis(result)

Generated SQL
SELECT category, SUM(sales) FROM sample___superstore GROUP BY category
KEY PERFORMANCE INDICATORS

SUM(sales)
Total   : 1,828,493.71
Average : 609,497.90
Maximum : 716,941.14
Minimum : 484,114.21


In [146]:
def detect_columns(df):

    numeric = list(

        df.select_dtypes(include=np.number).columns

    )

    categorical = list(

        df.select_dtypes(exclude=np.number).columns

    )

    return numeric, categorical

In [147]:
numeric, categorical = detect_columns(result)

print(numeric)

print(categorical)

['SUM(sales)']
['category']


In [148]:
def plot_bar(df):

    numeric, categorical = detect_columns(df)

    if len(numeric)==0 or len(categorical)==0:

        return

    fig = px.bar(

        df,

        x=categorical[0],

        y=numeric[0],

        title="Bar Chart"

    )

    fig.show()

In [149]:
plot_bar(result)

In [150]:
def plot_pie(df):

    numeric, categorical = detect_columns(df)

    if len(numeric)==0 or len(categorical)==0:

        return

    fig = px.pie(

        df,

        names=categorical[0],

        values=numeric[0],

        title="Pie Chart"

    )

    fig.show()

In [151]:
plot_pie(result)

In [152]:
def plot_line(df):

    numeric, categorical = detect_columns(df)

    if len(numeric)==0 or len(categorical)==0:

        return

    fig = px.line(

        df,

        x=categorical[0],

        y=numeric[0],

        markers=True,

        title="Line Chart"

    )

    fig.show()

In [153]:
def plot_scatter(df):

    numeric, categorical = detect_columns(df)

    if len(numeric) < 2:

        print("Need at least two numeric columns.")

        return

    fig = px.scatter(

        df,

        x=numeric[0],

        y=numeric[1],

        title="Scatter Plot"

    )

    fig.show()

In [154]:
def plot_histogram(df):

    numeric, _ = detect_columns(df)

    if len(numeric)==0:

        return

    fig = px.histogram(

        df,

        x=numeric[0],

        title="Histogram"

    )

    fig.show()

In [155]:
def plot_box(df):

    numeric, _ = detect_columns(df)

    if len(numeric)==0:

        return

    fig = px.box(

        df,

        y=numeric[0],

        title="Box Plot"

    )

    fig.show()

In [156]:
def auto_visualize(df):

    numeric, categorical = detect_columns(df)

    if len(numeric)==1 and len(categorical)==1:

        plot_bar(df)

        plot_pie(df)

        return

    elif len(numeric)>=2:

        plot_scatter(df)

        return

    elif len(numeric)==1:

        plot_histogram(df)

        return

    print("No suitable visualization found.")

In [157]:
result = ask_database(
    "Show total sales by category"
)

auto_visualize(result)

Generated SQL
SELECT category, SUM(sales) FROM sample___superstore GROUP BY category


In [158]:
def analyze(question):

    df = ask_database(question)

    print("\n")

    print("="*60)

    print("QUERY RESULT")

    print("="*60)

    display(df)

    print()

    show_kpis(df)

    print()

    auto_visualize(df)

    return df

In [159]:
analyze(
    "Show total sales by category"
)

Generated SQL
SELECT category, SUM(sales) FROM sample___superstore GROUP BY category


QUERY RESULT


,category,SUM(sales)
0,Furniture,484114.2085
1,Office Supplies,627438.3570
2,Technology,716941.1400



KEY PERFORMANCE INDICATORS

SUM(sales)
Total   : 1,828,493.71
Average : 609,497.90
Maximum : 716,941.14
Minimum : 484,114.21



,category,SUM(sales)
0,Furniture,484114.2085
1,Office Supplies,627438.3570
2,Technology,716941.1400


In [163]:
questions = [

    "Top 10 customers by sales",

    "Sales by region",

    "Profit by category",

    "Monthly sales",

    "Average discount"

]

for q in questions:

    print("="*80)

    print(q)
    print("="*80)

Top 10 customers by sales
Sales by region
Profit by category
Monthly sales
Average discount


In [164]:
result.to_csv(

    "exports/query_result.csv",

    index=False

)

print("Result saved successfully.")

Result saved successfully.


In [165]:
from google.colab import files

files.download("exports/query_result.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [166]:
import gradio as gr
import plotly.express as px
import pandas as pd
import sqlite3
import json
import numpy as np
import google.generativeai as genai
from getpass import getpass

In [167]:
api_key = getpass("Enter Gemini API Key: ")

genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

Enter Gemini API Key: ··········


In [168]:
connection = sqlite3.connect(
    "database/retail_database.db"
)

In [169]:
with open(
    "schema/database_schema.json",
    "r"
) as file:

    schema = json.load(file)

In [170]:
schema_text = ""

for table in schema:

    schema_text += f"\nTable: {table}\n"

    for column in schema[table]:

        schema_text += f"{column['column']} ({column['datatype']})\n"

In [171]:
SYSTEM_PROMPT = f"""
You are an expert SQLite SQL developer.

Database Schema:

{schema_text}

Rules:

Generate only SQL.

Use existing tables.

Use existing columns.

Do not explain.

Return executable SQL only.
"""

In [172]:
def generate_sql(question):

    prompt = SYSTEM_PROMPT + f"""

Question:

{question}

SQL:
"""

    response = model.generate_content(prompt)

    sql = response.text

    sql = sql.replace("```sql", "")

    sql = sql.replace("```", "")

    return sql.strip()

In [173]:
def execute_sql(sql):

    return pd.read_sql(sql, connection)

In [174]:
def generate_kpis(df):

    numeric = df.select_dtypes(include=np.number)

    if numeric.empty:

        return "No KPIs"

    text = ""

    for column in numeric.columns:

        text += f"""

{column}

Total : {numeric[column].sum():,.2f}

Average : {numeric[column].mean():,.2f}

Maximum : {numeric[column].max():,.2f}

Minimum : {numeric[column].min():,.2f}

-------------------------

"""

    return text

In [175]:
def generate_chart(df):

    numeric = list(

        df.select_dtypes(include=np.number).columns

    )

    categorical = list(

        df.select_dtypes(exclude=np.number).columns

    )

    if len(numeric)>=1 and len(categorical)>=1:

        fig = px.bar(

            df,

            x=categorical[0],

            y=numeric[0],

            title="AI Dashboard"

        )

        return fig

    elif len(numeric)>=2:

        fig = px.scatter(

            df,

            x=numeric[0],

            y=numeric[1]

        )

        return fig

    else:

        return px.histogram(df)

In [176]:
def ask_ai(question):

    try:

        sql = generate_sql(question)

        df = execute_sql(sql)

        kpis = generate_kpis(df)

        fig = generate_chart(df)

        return sql, df, kpis, fig

    except Exception as e:

        return str(e), pd.DataFrame(), "", None

In [177]:
demo = gr.Interface(

    fn=ask_ai,

    inputs=gr.Textbox(

        lines=2,

        placeholder="Ask a business question..."

    ),

    outputs=[

        gr.Textbox(label="Generated SQL"),

        gr.Dataframe(label="Query Result"),

        gr.Textbox(label="KPIs"),

        gr.Plot(label="Visualization")

    ],

    title="AI Dashboard Generator",

    description="Ask business questions in natural language."
)

In [179]:
response = model.generate_content("Hello")

print(response.text)

Hello! How can I help you today?


In [181]:
question = "show total sales by category"

sql = generate_sql(question)

print(sql)

ite
SELECT
  category,
  SUM(sales) AS total_sales
FROM sample___superstore
GROUP BY
  category;


In [184]:
def generate_sql(question):

    question = question.lower()

    sql_map = {

        "show total sales by category":
        """
        SELECT Category,
               SUM(Sales) AS TotalSales
        FROM sample___superstore
        GROUP BY Category
        """,

        "top 10 customers":
        """
        SELECT Customer_Name,
               SUM(Sales) AS TotalSales
        FROM sample___superstore
        GROUP BY Customer_Name
        ORDER BY TotalSales DESC
        LIMIT 10
        """,

        "sales by region":
        """
        SELECT Region,
               SUM(Sales) AS TotalSales
        FROM sample___superstore
        GROUP BY Region
        """
    }

    return sql_map.get(question, "SELECT * FROM sample___superstore LIMIT 10")

In [185]:
demo = gr.Interface(

    fn=ask_ai,

    inputs=gr.Textbox(

        lines=2,

        placeholder="Ask a business question..."

    ),

    outputs=[

        gr.Textbox(label="Generated SQL"),

        gr.Dataframe(label="Query Result"),

        gr.Textbox(label="KPIs"),

        gr.Plot(label="Visualization")

    ],

    title="AI Dashboard Generator",

    description="Ask business questions in natural language."
)

In [ ]:
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8a5e0bb09d50d40fd7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
